# Groupby

:::{admonition} Lesson Content
:class: note, dropdown

- 🔪 Groupby

:::

## Groupby

While we have 146 individual readings in our dataset sometimes we don't care about each individual reading.  Instead we probably care about the aggregate of a specific group of readings. 

For example:
* Given the average temperature of every county in the US, what is the average temperature in each state?
* Given a list of the opening dates of every Chuck E Cheese stores, how many Chuck E Cheeses were opened each year? 🧀

In pandas (and tabular data in general) we answer questions like that that with `groupby`.

### Breaking `groupby` into conceptual parts

In addition to the dataframe, there are three main parts to a groupby:
1. Which variable we want to group together
2. How we want to group
3. The variable we want to see in the end

Without getting into syntax yet we can start by identifiying these in our two example questions.

**Given the average temperature of every county in the US, what is the average temperature in each state?**

* _Which variable to group together?_ -> We want to group counties into states
* _How do we want to group?_ -> Take the average
* _What variable do we want to look at?_ Temperature

**Given a list of the opening dates of every Chuck E Cheese stores, how many Chuck E Cheeses were opened each year?**

* _Which variable to group together?_ -> We want to group individual days into years
* _How do we want to group?_ -> Count them
* _What variable do we want to look at?_ Number of stores

:::{admonition} 📝 Check your understanding
:class: tip

Identify each of three main groupby parts in the following scenario:

**Given the hourly temperatures for a location over the course of a month, what were the daily highs?**

1. _Which variable to group together?_
2. _How do we want to group?_
3. _What variable do we want to look at?_

:::

### Pre-processing

The code below adds to new columns "CO2_rating", and "CH4_rating" to the dataframe. These classify the $CO_2$ and $CO_4$ measurements qualitatively instead of quantitatively so we can experiment with using groupby. Don't worry too much about the code below, but notice the new columns at the end of the dataframe.

In [121]:
# Define cutoffs for the carbon dioxide and methane ratings
bins_co2 = [float('-inf'), 430, 440, float('inf')]
bins_ch4 = [float('-inf'), 2.0, 2.3, float('inf')]
labels = ['low', 'medium', 'high']

In [127]:
# Add carbon dioxide rating
picarro_flight['CO2_rating'] = pd.cut(
    picarro_flight[' CO2_ppm'], bins=bins_co2, 
    labels=labels, include_lowest=True
).astype('str')

In [134]:
picarro_flight['CH4_rating']

datetime (UTC)
2024-07-02 15:00:52      high
2024-07-02 15:00:54      high
2024-07-02 15:00:56      high
2024-07-02 15:00:58      high
2024-07-02 15:01:00      high
                        ...  
2024-07-02 17:57:28    medium
2024-07-02 17:57:30    medium
2024-07-02 17:57:32    medium
2024-07-02 17:57:34    medium
2024-07-02 17:57:36    medium
Name: CH4_rating, Length: 5303, dtype: object

In [133]:
# Add methane rating
picarro_flight['CH4_rating'] = pd.cut(
    picarro_flight[' CH4_ppm'], bins=bins_ch4, 
    labels=labels, include_lowest=True
).astype('str')

In [ ]:
# Drop unneeded columns for the next exercise
picarro_flight.drop(
    ['Time_Start', ' Time_Stop', ' GPS_Lon', ' GPS_Lat'], axis=1, inplace=True
)

### `groupby` syntax

We can take these `groupby` concepts and translate them into syntax.  The first two parts (which variable to group & how do we want to group) are required for pandas.  The third one is optional.

Starting with just the two required variables, the general syntax is:

`DATAFRAME.groupby(WHICH_GROUP).AGGREGATION()`

Words in all capitals are variables.  We'll go into each part a little more below.

In [111]:
picarro_flight

 CO2_ppm       float64
 CH4_ppm       float64
 H2O_ppm       float64
 CO_ppb        float64
 GPS_Alt       float64
CO2_rating    category
CH4_rating    category
dtype: object

In [137]:
picarro_flight.groupby("CO2_rating").mean(numeric_only=True)

,CO2_ppm,CH4_ppm,H2O_ppm,CO_ppb,GPS_Alt
CO2_rating,,,,,
high,450.436554,2.192665,15200.166311,196.034755,261.402532
low,426.565941,1.967308,6102.602509,103.925914,1078.394145
medium,433.416890,2.019003,12846.384051,146.918542,171.895462


What do we see in the output?
- Which (if any) of the other species are directly correlated with carbon dioxide?
- Are the highest $CO_2$ values near the surface or higher up?

#### `'WHICH_GROUP'`

This variable can be any of the columns in the dataframe that can be put into discrete piles.  We used `'safety_level'` because it can easily be put into piles -- `low`, `medium` and `high`.

We are still allowed to group on other columns, it's just that the practicality can become questionable.

In [138]:
picarro_flight.groupby("CH4_rating").min(numeric_only=True)

,CO2_ppm,CH4_ppm,H2O_ppm,CO_ppb,GPS_Alt
CH4_rating,,,,,
high,440.061,2.30336,10057.0,131.0,49.318883
low,419.591,1.92809,1080.0,57.3,18.900000
medium,425.214,2.00006,4060.0,97.8,-7.338963


:::{note}

Because they are right next to each other it can be really tempting to want to put the variable you are most interested in seeing in place of `WHICH_GROUP`.  If you do this and use a column that can't be grouped, so consider the `WHICH_GROUP` variable if you are debugging.

:::

#### Aggregation Functions 

The functions you can use in a groupby are limited, but there are still lots of options.  Common ones include:
* `.count()` - find the total number of rows
* `.min()`- find the minimum value of those rows
* `.max()` - find the maximum value of those rows
* `.mean()`- find the mean value of those rows
* `.sum()` - find the sum of the values of those rows

In [142]:
picarro_flight.groupby("CH4_rating").count()

,CO2_ppm,CH4_ppm,H2O_ppm,CO_ppb,GPS_Alt,CO2_rating
CH4_rating,,,,,,
high,89,89,89,89,89,89
low,3072,3072,3072,3072,3072,3072
medium,2142,2142,2142,2142,2142,2142


#### The third input

In the conceptual explanation we often also specify which variable we want to look at.  In `pandas` this isn't always as important, since by default it gives us the full dataframe.  If for some reason we do need to specify the exact variable of interest (ex. we have a very large dataframe, we are making a plot), we do that by adding the column name in after the groupby.

`DATAFRAME.groupby(WHICH_GROUP)[INTEREST_VAR].AGGREGATION()`

In [140]:
# Group our water dataframe by saftey level and take the mean of the discharge values for each safety level group
picarro_flight.groupby("CO2_rating")[' CH4_ppm'].mean()

CO2_rating
high      2.192665
low       1.967308
medium    2.019003
Name:  CH4_ppm, dtype: float64

:::{admonition} 📝 Check your understanding
:class: tip

1. What is the maximum pH value during low safety level?

2. What is the mean discharge when dam releases are happening?

:::

### Breaking down the process

There is a lot that happens in a single step with `groupby` and it can be a lot to take in.  One way to mentally situate this process is to think about **split-apply-combine**.

**split-apply-combine** breaks down the `groupby` process into those three steps:
1. SPLIT the full data set into groups.  Split is related to the question _Which variable to group together?_
2. APPLY the aggregation function to the individual groups.  Apply is related to the question _How do we want to group?_
3. COMBINE the aggregated data into a new dataframe


<img src="https://static.packt-cdn.com/products/9781783985128/graphics/5128OS_09_01.jpg" width=550>




